In [0]:
# --- Imports
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, unix_timestamp, round, when, expr, lit, current_timestamp, to_json, struct
)
from pyspark.sql.utils import AnalysisException
import uuid

In [0]:
# Key columns to help identify a row in logs (adjust to your dataset)
KEY_COLS = ["vendor_id", "pickup_datetime"]

# Data-quality rules (add/edit freely)
DQ_RULES = [
    # Simple threshold rule: column + operator + threshold
    {"name": "fare_too_high", "column": "fare_amount_usd", "op": ">", "threshold": 500.0, "severity": "WARN"},
    {"name": "distance_too_high_km", "column": "trip_distance_km", "op": ">", "threshold": 150.0, "severity": "ERROR"},
    # Free-form expression (great for compound or null checks)
    {"name": "duration_outlier", "expr": "trip_duration_min > 240 OR trip_duration_min < 0", "severity": "ERROR"},
]

# ------------------------
# Utility: create tables if missing
# ------------------------
def init_tables(spark: SparkSession):
    # Exception log: one row per (violating row x rule)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            job_run_id   STRING,
            rule_name    STRING,
            rule_severity STRING,
            rule_type    STRING,
            column_name  STRING,
            threshold    STRING,
            actual_value STRING,
            key_json     STRING,
            source_table STRING,
            target_table STRING,
            stage        STRING,
            event_ts     TIMESTAMP
        ) USING DELTA
    """)
    # Quarantine holds the **full** rows that fail ERROR-level rules
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE}
        USING DELTA AS SELECT * FROM (SELECT 1 AS dummy) WHERE 1=0
    """)

In [0]:
# ------------------------
# 1. EXTRACT
# ------------------------
def extract_data(spark: SparkSession, source: str) -> DataFrame:
    """
    Extract data from Unity Catalog table or fallback sample dataset.
    Error handling:
    - Catch table not found / schema issues.
    - Fallback to a small sample DataFrame for practice.
    """
    try:
        df = spark.table(source)
        print(f"✅ Loaded data from {source} with {df.count()} rows.")
    except AnalysisException as ae:
        print(f"⚠️ Table {source} not found. Using fallback dataset. Details: {ae}")
        from pyspark.sql.types import StructType, StructField, StringType, DoubleType
        data = [
            ("VTS001", "2024-05-01 08:00:00", "2024-05-01 08:15:00", 3.5, 12.0, "credit_card"),
            ("VTS002", "2024-05-01 09:30:00", "2024-05-01 09:50:00", 5.0, 18.5, "cash"),
            ("VTS003", "2024-05-01 10:00:00", "2024-05-01 10:25:00", 7.0, 24.0, None),
            ("VTS004", None, "2024-05-01 11:15:00", 2.0, 8.0, "credit_card"),
        ]
        schema = StructType([
            StructField("vendor_id", StringType(), True),
            StructField("pickup_datetime", StringType(), True),
            StructField("dropoff_datetime", StringType(), True),
            StructField("trip_distance_miles", DoubleType(), True),
            StructField("fare_amount_usd", DoubleType(), True),
            StructField("payment_type", StringType(), True)
        ])
        df = spark.createDataFrame(data, schema)
    except Exception as e:
        raise RuntimeError(f"❌ Unexpected error during extraction: {e}")
    
    return df

In [0]:
# ------------------------
# 2. TRANSFORM
# ------------------------
def transform_data(df: DataFrame) -> DataFrame:
    """
    Transform data by cleaning, enriching, and standardizing columns.
    Error handling:
    - Validate required columns exist.
    - Catch calculation errors (e.g., null date parsing).
    """
    required_cols = ["pickup_datetime", "dropoff_datetime", "trip_distance_miles", "fare_amount_usd"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"❌ Missing required columns: {missing_cols}")

    try:
        df_transformed = (
            df.filter(col("pickup_datetime").isNotNull())
              .withColumn("trip_distance_km", round(col("trip_distance_miles") * 1.60934, 2))
              .withColumn("trip_duration_min",
                          round((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 1))
              .withColumn("payment_type", when(col("payment_type").isNull(), "unknown").otherwise(col("payment_type")))
        )
    except Exception as e:
        raise RuntimeError(f"❌ Error during transformation: {e}")
    
    # Quick data sanity check
    if df_transformed.filter(col("trip_distance_km") < 0).count() > 0:
        raise ValueError("❌ Negative trip distances found after transformation.")

    return df_transformed

In [0]:
# ------------------------
# Data-Quality: evaluate rules, log violations, quarantine ERRORs
# ------------------------
def apply_quality_checks_and_log(
    spark: SparkSession,
    df: DataFrame,
    rules: list,
    job_run_id: str,
    source_table: str,
    target_table: str,
    stage: str = "validation",
    key_cols: list = None,
    quarantine_table: str = None,
    log_table: str = None
) -> DataFrame:
    """
    - Builds a single violations DF across all rules.
    - Appends violations to LOG_TABLE.
    - Appends ERROR rows (full records) to QUARANTINE_TABLE and removes them from df.
    Returns the cleaned df that will continue to LOAD.
    """
    key_cols = key_cols or []
    log_table = log_table or LOG_TABLE
    quarantine_table = quarantine_table or QUARANTINE_TABLE

    # Build one violations DF (stack of all rule violations)
    violations_union = None
    error_union_rows = None

    for r in rules:
        rule_name = r["name"]
        severity  = r.get("severity", "ERROR").upper()
        if "expr" in r:
            cond_str = r["expr"]
            rule_type = "expr"
            col_name  = None
            threshold = None
            expr_str  = cond_str
        else:
            rule_type = "threshold"
            col_name  = r["column"]
            op        = r.get("op", ">")
            threshold = r["threshold"]
            expr_str  = f"`{col_name}` {op} {threshold}"

        # Rows that violate this rule
        viol_rows = (
            df.filter(expr(expr_str))
              .withColumn("job_run_id", lit(job_run_id))
              .withColumn("rule_name", lit(rule_name))
              .withColumn("rule_severity", lit(severity))
              .withColumn("rule_type", lit(rule_type))
              .withColumn("column_name", lit(col_name))
              .withColumn("threshold", lit(str(threshold) if threshold is not None else None))
              .withColumn("actual_value", 
                          # if column present, log the actual, else None (expr case)
                          when(col_name.isNotNull() & (col_name != ""), col(col_name).cast("string")).otherwise(lit(None)))
              .withColumn("key_json", to_json(struct(*[col(c) for c in key_cols])) if key_cols else lit(None))
              .withColumn("source_table", lit(source_table))
              .withColumn("target_table", lit(target_table))
              .withColumn("stage", lit(stage))
              .withColumn("event_ts", current_timestamp())
              # Select log schema columns only for the log write
              .select("job_run_id","rule_name","rule_severity","rule_type","column_name",
                      "threshold","actual_value","key_json","source_table","target_table","stage","event_ts")
        )

        violations_union = viol_rows if violations_union is None else violations_union.unionByName(viol_rows)

        # For ERROR rules, collect the full offending rows to quarantine
        if severity == "ERROR":
            err_rows = df.filter(expr(expr_str))
            error_union_rows = err_rows if error_union_rows is None else error_union_rows.unionByName(err_rows)

    # 1) Append to LOG table (if there were any violations)
    if violations_union is not None and violations_union.head(1):
        violations_union.write.format("delta").mode("append").saveAsTable(log_table)
        print(f"📝 Logged {violations_union.count()} violating rows into {log_table}")

    # 2) Quarantine ERROR rows and remove them from df
    cleaned_df = df
    if error_union_rows is not None and error_union_rows.head(1):
        # Drop duplicates to avoid multiple-logging same row across rules in quarantine
        error_union_rows = error_union_rows.dropDuplicates()
        error_union_rows.write.format("delta").mode("append").saveAsTable(quarantine_table)
        print(f"🚧 Quarantined {error_union_rows.count()} rows into {quarantine_table}")

        # Remove quarantined rows from the pipeline output
        # Build an anti-join using key columns if available; otherwise, use all columns to de-dup (heavy)
        if KEY_COLS:
            cleaned_df = cleaned_df.join(error_union_rows.select(*KEY_COLS).dropDuplicates(), on=KEY_COLS, how="left_anti")
        else:
            cleaned_df = cleaned_df.join(error_union_rows.dropDuplicates(), on=list(df.columns), how="left_anti")

    return cleaned_df

In [0]:
# ------------------------
# 3. LOAD
# ------------------------
def load_data(df: DataFrame, target: str):
    """
    Load data into Delta table in Unity Catalog.
    Error handling:
    - Catch write permission errors.
    - Validate row count before loading.
    """
    if df.count() == 0:
        raise ValueError("❌ No data to load. Aborting write operation.")

    try:
        df.write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"✅ Data loaded into {target} ({df.count()} rows).")
    except AnalysisException as ae:
        raise RuntimeError(f"❌ Could not write to target table {target}: {ae}")
    except Exception as e:
        raise RuntimeError(f"❌ Unexpected error during load: {e}")

In [0]:
# ------------------------
# Pipeline runner
# ------------------------
def run_pipeline():
    spark = SparkSession.builder.getOrCreate()
    init_tables(spark)

    job_run_id = str(uuid.uuid4())
    print(f"🏁 Starting ETL job_run_id={job_run_id}")

    try:
        # Extract
        df_raw = extract_data(spark, SOURCE_TABLE)

        # Transform
        df_tx = transform_data(df_raw)

        # Data Quality + Logging + Quarantine
        df_clean = apply_quality_checks_and_log(
            spark=spark,
            df=df_tx,
            rules=DQ_RULES,
            job_run_id=job_run_id,
            source_table=SOURCE_TABLE,
            target_table=TARGET_TABLE,
            key_cols=KEY_COLS,
            quarantine_table=QUARANTINE_TABLE,
            log_table=LOG_TABLE
        )

        # Load
        load_data(df_clean, TARGET_TABLE)

        print("✅ Pipeline finished successfully.")
        # Quick peek
        # spark.table(TARGET_TABLE).display()
        # spark.table(LOG_TABLE).orderBy(col("event_ts").desc()).display()
        # spark.table(QUARANTINE_TABLE).display()

    except Exception as e:
        print(f"❌ Pipeline failed (job_run_id={job_run_id}): {e}")
        raise

In [0]:
# ------------------------
# 5. EXECUTE
# ------------------------
if __name__ == "__main__":
    run_pipeline()